In [1]:
%load_ext autoreload
%autoreload 2

from fibsem import utils, acquire, movement

from fibsem.applications.autolamella.structures import (
    AutoLamellaTaskProtocol,
    AutoLamellaWorkflowConfig,
    AutoLamellaWorkflowOptions,
    Experiment,
    Lamella,
    DefectState,
)
from fibsem.applications.autolamella.workflows.tasks.tasks import run_tasks

from fibsem.structures import FibsemStagePosition, FibsemImage,MicroscopeState
from fibsem.applications.autolamella.workflows.tasks.tasks import MillTrenchTask, MillTrenchTaskConfig

Default configuration default-configuration. Configuration Path: C:\Users\rohit\Documents\Code\newFIBSEM\fibsem-os-NYSBC\fibsem\config\microscope-configuration.yaml


c:\Users\rohit\.conda\envs\fibsem\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# connect to microscope
microscope, settings = utils.setup_session(manufacturer="Demo", ip_address="localhost")

2026-03-04 16:51:10,374 — root — INFO — _setup_image_iterators:466 — SEM or FIB data path not configured in simulator settings, using random noise generation
2026-03-04 16:51:10,384 — root — INFO — connect_to_microscope:282 — Microscope client connected to DemoMicroscope with serial number 123456 and software version 0.1
2026-03-04 16:51:10,394 — root — INFO — setup_session:325 — Finished setup for session: demo_2026-03-04-04-51-10PM


In [3]:
liftout_path = r"C:\Users\rohit\Documents\Code\newFIBSEM\liftout_method"


liftout_experiment = Experiment(liftout_path, "liftout_experiment")

In [4]:
microscope_state = microscope.get_microscope_state()

In [10]:
task_protocol = AutoLamellaTaskProtocol.load(r"C:\Users\rohit\Documents\Code\newFIBSEM\liftout-protocol.yaml")

In [11]:

print(task_protocol.workflow_config.tasks)


[AutoLamellaTaskDescription(name='Trench Milling', supervise=True, required=False, requires=[]), AutoLamellaTaskDescription(name='Undercut Milling', supervise=True, required=True, requires=[])]


In [12]:
landing_positions = liftout_experiment.landing_positions

for i in range(5):
    position = FibsemStagePosition(x=float(i), y=0, z=0)
    landing_positions.append(position)

print(liftout_experiment.landing_positions)


[FibsemStagePosition(name=None, x=0.0, y=0, z=0, r=None, t=None, coordinate_system=None), FibsemStagePosition(name=None, x=1.0, y=0, z=0, r=None, t=None, coordinate_system=None), FibsemStagePosition(name=None, x=2.0, y=0, z=0, r=None, t=None, coordinate_system=None), FibsemStagePosition(name=None, x=3.0, y=0, z=0, r=None, t=None, coordinate_system=None), FibsemStagePosition(name=None, x=4.0, y=0, z=0, r=None, t=None, coordinate_system=None)]


In [13]:
liftout_experiment.task_protocol = task_protocol

In [14]:
liftout_experiment.add_new_lamella(microscope_state,liftout_experiment.task_protocol.task_config,"Lamella01")

2026-03-04 17:02:36,872 — root — INFO — add_new_lamella:1001 — Created new lamella Lamella01 at C:\Users\rohit\Documents\Code\newFIBSEM\liftout_method\liftout_experiment\Lamella01
2026-03-04 17:02:36,891 — root — INFO — add_lamella:946 — Added lamella Lamella01 to experiment liftout_experiment


In [ ]:
## make dictionary of tasks to run

## tasks 

# 1 Milling Landing Sites

# 2 Milling Main Lamella Block

# these 2 for now

In [16]:
task_names = [t.name for t in liftout_experiment.task_protocol.workflow_config.tasks]
print(task_names)

['Trench Milling', 'Undercut Milling']


In [20]:
positions = liftout_experiment.positions
# print(positions)
lam1 = positions[0]
print(lam1)

Lamella(path=WindowsPath('C:/Users/rohit/Documents/Code/newFIBSEM/liftout_method/liftout_experiment/Lamella01'), number=1, petname='Lamella01', alignment_area=FibsemRectangle(left=0.7, top=0.3, width=0.25, height=0.4), _id='ed9c19ef-aa3b-4334-b183-e8fad47f842c', task_config=EventedDict({'Trench Milling': MillTrenchTaskConfig(task_name='Trench Milling', milling={'trench': FibsemMillingTaskConfig(name='Trench Milling', field_of_view=0.00075, channel=<BeamType.ION: 2>, alignment=MillingAlignment(enabled=True, interval_enabled=False, interval=30, rect=FibsemRectangle(left=0.7, top=0.3, width=0.25, height=0.4), use_autocontrast=True, use_autofocus=False, steps=3, imaging=ImageSettings(resolution=[1536, 1024], dwell_time=1e-06, hfw=0.00015, autocontrast=False, beam_type=<BeamType.ELECTRON: 1>, save=False, filename='default_image', autogamma=False, path='C:\\Users\\rohit', reduced_area=None, line_integration=None, scan_interlacing=None, frame_integration=None, drift_correction=False)), acquis

In [ ]:

task_name = "Trench Milling"

task_config = liftout_experiment.task_protocol.task_config[task_name]
task_config.parameters
task = MillTrenchTask(microscope=microscope, config=task_config,lamella=liftout_experiment.positions[0])

In [26]:
task.task_type

'MILL_TRENCH'

In [24]:
for task_name in task_names:

    if task_name == "Trench Milling":

        # do landing positions

        for landing_position in liftout_experiment.landing_positions:

            task_config = liftout_experiment.task_protocol.task_config[task_name]

            task = MillTrenchTask(microscope=microscope, config=task_config,lamella=liftout_experiment.positions[0])

            task.run()

            # run_tasks(microscope, [task], liftout_experiment)

2026-03-04 17:52:21,982 — root — INFO — pre_task:334 — Running Trench Milling, MILL_TRENCH (c4f67ce2-1d56-4fca-b7dd-73eedea31115) for Lamella01 (ed9c19ef-aa3b-4334-b183-e8fad47f842c)
2026-03-04 17:52:21,987 — root — INFO — update_status_ui:196 — Lamella01 [Trench Milling] Started
2026-03-04 17:52:21,990 — root — INFO — update_status_ui:196 — Lamella01 [Trench Milling] Moving to Trench Position...
2026-03-04 17:52:21,996 — root — INFO — get_target_position:1083 — Getting target position for FIB from MILLING
2026-03-04 17:52:22,002 — root — INFO — _safe_rotation_movement:2322 — tilting to flat for large rotation.
2026-03-04 17:52:22,008 — root — INFO — update_status_ui:196 — Lamella01 [Trench Milling] Preparing to Mill Trench...
2026-03-04 17:52:22,010 — root — INFO — update_status_ui:196 — Lamella01 [Trench Milling] Acquiring Reference Images...
2026-03-04 17:52:22,014 — root — INFO — acquire_image:344 — acquiring new ELECTRON image.
2026-03-04 17:52:26,854 — root — INFO — acquire_image